# ✈️ Airline Satisfaction Prediction

## 🎯 Learning Objectives
- Binary classification with mixed data types
- OneHotEncoder for categorical features
- StandardScaler for numerical features
- Handling missing data strategically
- Comparing simple vs ensemble models
- Evaluating beyond accuracy

## 🏢 Business Context
Airlines need to predict customer satisfaction to:
- Identify at-risk customers before they churn
- Improve services based on satisfaction drivers
- Allocate resources to high-impact improvements

## 📊 The Data
- **Source**: Airline customer feedback survey
- **Size**: ~100,000 passengers
- **Task**: Predict satisfied vs dissatisfied
- **Features**: Demographics, flight details, service ratings

👇 Run the cell below to load the data

In [ ]:
import pandas as pd

url = "https://wagon-public-datasets.s3.amazonaws.com/data-analytics/machine-learning/airline_satisfaction.csv"

airline_df = pd.read_csv(url)

airline_df.head()

## 📊 Data Exploration

Before preprocessing, let's understand our data:

**Tasks:**
1. Check the shape (rows × columns)
2. Identify feature data types
3. Check for missing values
4. Examine target class balance

In [ ]:
airline_df.info()

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

Our target and some of our features are not yet ready to be used in a model.

Start by removing any columns that you think are unnecessary:

<details>
    <summary><i>Answer:</i></summary>
     
The `'Unnamed: 0'` column is an index column left over from when the dataset was exported as a CSV file — we can remove this.

Also, the `'id'` column will not provide any information relevant to our model, so this can be removed.
</details>       

In [ ]:
# YOUR CODE HERE

#### Handling Missing Values

We have missing data in `Arrival Delay in Minutes`.

**Question**: Should we use mean, median, or mode?

<details>
    <summary><i>Answer</i></summary>

- **Mean**: Sensitive to outliers (extreme delays skew average)
- **Median**: Robust to outliers (better for skewed distributions)
- **Mode**: Only for categorical data

Flight delays are typically **right-skewed** (most flights on time, few very delayed).
Median is the best choice! ✅
</details>   

In [ ]:
# YOUR CODE HERE

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'data_cleaning',
    n_rows=len(airline_df),
    has_unnamed='Unnamed: 0' in airline_df.columns,
    has_id='id' in airline_df.columns,
    missing_count=airline_df.isnull().sum().sum()
)
result.write()
print(result.check())

#### Encoding the Target

Our target `satisfaction` has two string values. We need to convert to 0/1.

There are two approaches we could take to this: 
1. **LabelEncoder**: `fit_transform()` (what we'll use)
2. **Manual mapping**: `map({'satisfied': 1, 'neutral or dissatisfied': 0})`

Why use LabelEncoder? We can save it and decode predictions later!

Save the target as `airline_target`

In [ ]:
# YOUR CODE HERE

#### Train-Test Split

**Important**: Use `stratify` to preserve class balance in both sets!

Without stratification train might be 60% satisfied and test 40% satisfied 

With stratification both train and test will match the original 57% vs 43% 

In [ ]:
# YOUR CODE HERE

### Preprocessing

Next step is to preprocess our data.
Let's deal with the categorical features in our dataset first.

A number of these contain string labels that must be transformed into numeric values so that they can be understood by the model that we end up training.

Encode the following columns `['Gender', 'Customer Type', 'Type of Travel', 'Class']` as a dataframe called 'cat_features'. Consider trying to use sklearn's `OneHotEncoder` to do this!

**Variables to create:**
- `X_train_cat_encoded` - Encoded training features
- `X_test_cat_encoded` - Encoded test features

<details>
    <summary><i>Hint 1:</i></summary>

Be sure to check how many labels each column has. If any columns are binary, you can use the parameter `drop='if_binary'` in your One Hot Encoder to preserve only one column if there are two labels.

**Why?** If a column has only 2 values (e.g., Male/Female), one column is enough:
- Female=0 means Male=1
- Female=1 means Male=0

Two columns would be redundant (perfect negative correlation)!

</details>  

<details>
    <summary><i>Hint 2:</i></summary>

If the output of your One Hot Encoder doesn't look right, check the 'sparse_output' parameter of your encoder.
</details>  


<details>
    <summary><i>Hint 3:</i></summary>

You can extract the names for your columns with the `.get_feature_names_out()` method of your One Hot Encoder.
</details>

In [ ]:
# YOUR CODE HERE

Next we will address the numeric features in our dataset.

Create dataframes that contain the scaled numeric features.

**Variables to create:**
- `X_train_num_scaled` - Scaled training features
- `X_test_num_scaled` - Scaled test features

<details>
    <summary><i>Hint 1:</i></summary>

Remember that you can use `.select_dtypes()` to select only columns of a specific data type from a dataframe!
</details>  


In [ ]:
# YOUR CODE HERE

Now that we have processed our numeric and categorical features, let's combine them back into one variable called 'X_train_preprocessed' and 'X_test_preprocessed' that contains **all** of our **preprocessed features** for each train and test data.

<details>
    <summary><i>Hint 1:</i></summary>

`pd.concat()` (The 'axis' parameter is important!)
</details>  

In [ ]:
# YOUR CODE HERE

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'preprocessing',
    target_exists='airline_target' in locals(),
    train_size=len(X_train),
    test_size=len(X_test),
    train_features=len(X_train_preprocessed.columns),
    test_features=len(X_test_preprocessed.columns)
)
result.write()
print(result.check())

Well done! That's our preprocessing done. It's not a super quick process, but it is **very** important.

Now, let's create our models.

## Modeling

We will start with a simple classifier. Train a logistic regression model on your train data. Then evaluate it's accuracy score on your test data!

In [ ]:
# YOUR CODE HERE

### Evaluating the Baseline Model

Remember, accuracy alone doesn't tell the full story! Let's take a look at the precision, recall, and F1 scores.

In [ ]:
# YOUR CODE HERE

These are pretty fantastic scores, but let's see if we can do better...

Let's try using a more complex model on our preprocessed data. A [random forest classifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) might be a good alternative choice.

<details>
<summary><i>Note</i></summary>

Random Forest models reduce overfitting by averaging multiple decision trees and is less sensitive to noise and outliers in the data.
</details>

Try instantiating and fitting a random forest classifier model with the same data as above and compare the accuracy scores.

In [ ]:
# your code here
from sklearn.ensemble import RandomForestClassifier

forest_model = RandomForestClassifier()

forest_model.fit(X_train_preprocessed, y_train)

forest_model.score(X_test_preprocessed, y_test)

### 🔍 Feature Importance Analysis

Random Forest tells us which features matter most for predicting satisfaction!

In [ ]:
# Extract feature importance
feature_importance = pd.DataFrame({
    'Feature': X_train_preprocessed.columns,
    'Importance': forest_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Visualize top 10
import plotly.express as px

fig = px.bar(
    feature_importance.head(10),
    x='Importance',
    y='Feature',
    orientation='h',
    title='Top 10 Most Important Features for Predicting Satisfaction'
)
fig.show()

# Print top 5
print("🏆 Top 5 Most Important Features:")
for i, (idx, row) in enumerate(feature_importance.head(5).iterrows(), 1):
    print(f"{i}. {row['Feature']}: {row['Importance']:.4f}")

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'models',
    lr_trained=hasattr(model, 'coef_'),
    rf_trained=hasattr(forest_model, 'estimators_'),
    lr_accuracy=model.score(X_test_preprocessed, y_test),
    rf_accuracy=forest_model.score(X_test_preprocessed, y_test)
)
result.write()
print(result.check())

Your random forest classifier should have an **accuracy score of around 96%**, which is a substantial improvement on the simple model we ran earlier!

Certain models will perform better than others depending on your data. Knowing which one to use intuitively is a matter of practice. For example, random forest models are good for data that is *non-linear* in nature.

*Optional extra: pick one of the models in the documentation and use it to predict airline customer satisfaction!*

## 🏁 Conclusion

### 🏆 Key Takeaways

1. **Preprocessing Matters**
   - Mixed data types require different strategies
   - One-hot encoding for categorical features
   - StandardScaler for numerical features
   - Always fit on train, transform on test!

2. **Model Selection**
   - Logistic Regression: ~88% accuracy (good baseline)
   - Random Forest: ~96% accuracy (substantial improvement!)
   - Ensemble methods handle non-linear relationships better

3. **Evaluation Beyond Accuracy**
   - Confusion matrix shows where models fail
   - Precision/Recall trade-offs matter
   - Feature importance reveals satisfaction drivers

### 📚 Resources

- [Sklearn Classification Models](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html)
- [Feature Engineering Guide](https://scikit-learn.org/stable/modules/preprocessing.html)